# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GraphQL enrichment (`pr_title`, `pr_body`, `repo_star_count`, `is_resolved`) → unified code enrichment (compare patches, base/snapshot zipballs, patched content, dependency resolution).

Checkpoints are written as zstd-compressed JSON under `data/checkpoints/` (`*.json.zst`). The flattened modeling export is `data/exports/dataset.parquet` (zstd Parquet).

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [ ]:
import asyncio
import logging

import aiohttp
import pandas as pd
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    config,
    gh_archive,
    github_api,
    github_graphql,
    parquet_export,
)


load_dotenv()
logging.basicConfig(level=logging.INFO)

In [ ]:
gh_archive_semaphore = asyncio.Semaphore(config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(config.GITHUB_API_CONCURRENCY)

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GH_ARCHIVE_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        config.RANGE_START,
        config.RANGE_END,
        gh_archive_semaphore,
    )

In [ ]:
checkpoints.save_dataset_checkpoint(dataset, config.DATASET_CHECKPOINT_RAW_PATH)

In [ ]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset,
    config.SNAPSHOT_COMMITS_TO_KEEP,
)

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GITHUB_API_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    await github_graphql.enrich_dataset_with_graphql_info(dataset, session, gh_semaphore)

In [ ]:
checkpoints.save_dataset_checkpoint(dataset, "../data/checkpoints/checkpoint_pr_enriched.json.zst")

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GITHUB_API_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    await github_api.enrich_dataset_with_code(dataset, session, gh_semaphore)

In [ ]:
checkpoints.save_dataset_checkpoint(dataset, config.DATASET_CHECKPOINT_FINAL_PATH)

In [ ]:
# Compute dataset statistics
num_prs = 0
num_snapshot_commits = 0
num_files_with_comments = 0
num_files_without_comments = 0
num_resolved_comments = 0
num_unresolved_comments = 0
num_files_with_outgoing_deps = 0
total_outgoing_dep_files = 0
num_files_with_incoming_deps = 0
total_incoming_dep_files = 0
num_commits_with_metadata = 0
total_metadata_files = 0
num_commits_with_file_tree = 0

for pr_map in dataset.values():
    for pr_entry in pr_map.values():
        num_prs += 1
        for path_map in pr_entry["commits"].values():
            num_snapshot_commits += 1
            snap_with = 0
            snap_without = 0
            for path, file_entry in path_map.items():
                if path == "metadata_files":
                    num_commits_with_metadata += 1
                    total_metadata_files += len(file_entry)
                    continue
                if path == "file_tree":
                    tree_text = file_entry.get("tree", "")
                    if tree_text:
                        num_commits_with_file_tree += 1
                    continue
                comments = file_entry.get("comments", [])
                for comment in comments:
                    if comment.get("is_resolved", False):
                        num_resolved_comments += 1
                    else:
                        num_unresolved_comments += 1
                if comments:
                    snap_with += 1
                else:
                    snap_without += 1
                out_deps = file_entry.get("outgoing_dependencies", {})
                if out_deps:
                    num_files_with_outgoing_deps += 1
                    total_outgoing_dep_files += len(out_deps)
                in_deps = file_entry.get("incoming_dependencies", {})
                if in_deps:
                    num_files_with_incoming_deps += 1
                    total_incoming_dep_files += len(in_deps)
            num_files_with_comments += snap_with
            num_files_without_comments += snap_without

num_files = num_files_with_comments + num_files_without_comments
num_comments = num_resolved_comments + num_unresolved_comments

logging.info("Number of PRs:                  %d", num_prs)
logging.info("Number of snapshot commits:     %d", num_snapshot_commits)
logging.info("Number of files (total):        %d", num_files)
logging.info("  - with comments:              %d", num_files_with_comments)
logging.info("  - without comments:           %d", num_files_without_comments)
logging.info("Number of comments (total):     %d", num_comments)
logging.info("  - resolved:                   %d", num_resolved_comments)
logging.info("  - unresolved:                 %d", num_unresolved_comments)

out_dep_pct = 100 * num_files_with_outgoing_deps / num_files if num_files else 0.0
logging.info("Files with >=1 outgoing dep:    %d (%.1f%%)", num_files_with_outgoing_deps, out_dep_pct)
logging.info("Total outgoing dep entries:     %d", total_outgoing_dep_files)

in_dep_pct = 100 * num_files_with_incoming_deps / num_files if num_files else 0.0
logging.info("Files with >=1 incoming dep:    %d (%.1f%%)", num_files_with_incoming_deps, in_dep_pct)
logging.info("Total incoming dep entries:     %d", total_incoming_dep_files)

meta_pct = 100 * num_commits_with_metadata / num_snapshot_commits if num_snapshot_commits else 0.0
logging.info("Commits with metadata files:    %d (%.1f%%)", num_commits_with_metadata, meta_pct)
logging.info("Total metadata file entries:    %d", total_metadata_files)

file_tree_pct = 100 * num_commits_with_file_tree / num_snapshot_commits if num_snapshot_commits else 0.0
logging.info("Commits with file tree:         %d (%.1f%%)", num_commits_with_file_tree, file_tree_pct)

In [ ]:
out_path, stats = parquet_export.save_files_list_parquet(dataset, config.DATASET_PATH)

In [ ]:
df = pd.read_parquet(config.DATASET_PATH, engine="pyarrow")